In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "Fraud-detection-ML-V2" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
SRC_DIR = PROJECT_ROOT / "src"

TRAIN_FEATURE_PATH = PROCESSED_DATA_DIR / "ml_training_features.csv"
TEST_FEATURE_PATH = PROCESSED_DATA_DIR / "ml_test_features.csv"

if not TRAIN_FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Training feature file not found:\n{TRAIN_FEATURE_PATH}"
    )

if not TEST_FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Test feature file not found:\n{TEST_FEATURE_PATH}"
    )

df = pd.read_csv(TRAIN_FEATURE_PATH)
test_df = pd.read_csv(TEST_FEATURE_PATH)

RULE_CONFIG = {
    "amount_vs_avg_ratio_threshold": 3.0,
    "velocity_5min_threshold": 5,
    "impossible_travel_distance_km": 500.0,
    "impossible_travel_time_sec": 3600.0,
    "rule_weights": {
        "HIGH_AMOUNT": 0.25,
        "HIGH_VELOCITY": 0.25,
        "IMPOSSIBLE_TRAVEL": 0.25,
        "NEW_MERCHANT_CATEGORY": 0.25
    }
}

FEATURE_COLUMNS = [
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user"
]

print("========== PHASE 9 INITIALIZATION ==========")
print("Training rows:", len(df))
print("Test rows:", len(test_df))
print("Number of rule features:", len(FEATURE_COLUMNS))
print("Rule configuration loaded:", True)
print("============================================")

========== PHASE 9 INITIALIZATION ==========
Training rows: 1296675
Test rows: 555719
Number of rule features: 6
Rule configuration loaded: True


In [2]:
required_columns = FEATURE_COLUMNS

missing_training = [
    column
    for column in required_columns
    if column not in df.columns
]

missing_test = [
    column
    for column in required_columns
    if column not in test_df.columns
]

if missing_training:
    raise ValueError(
        f"Missing training rule fields: {missing_training}"
    )

if missing_test:
    raise ValueError(
        f"Missing test rule fields: {missing_test}"
    )

for column in FEATURE_COLUMNS:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )
    test_df[column] = pd.to_numeric(
        test_df[column],
        errors="coerce"
    )

if df[FEATURE_COLUMNS].isnull().any().any():
    raise ValueError(
        "Training data contains invalid rule fields."
    )

if test_df[FEATURE_COLUMNS].isnull().any().any():
    raise ValueError(
        "Test data contains invalid rule fields."
    )

if not np.isfinite(
    df[FEATURE_COLUMNS].to_numpy(dtype=float)
).all():
    raise ValueError(
        "Training rule fields contain non-finite values."
    )

if not np.isfinite(
    test_df[FEATURE_COLUMNS].to_numpy(dtype=float)
).all():
    raise ValueError(
        "Test rule fields contain non-finite values."
    )

print("========== RULE INPUT VALIDATION ==========")
print("All required fields present:", True)
print("Training values valid:", True)
print("Test values valid:", True)
print("===========================================")

========== RULE INPUT VALIDATION ==========
All required fields present: True
Training values valid: True
Test values valid: True


In [3]:
def amount_threshold_rule(transaction, config):
    return (
        transaction["amount_vs_avg_ratio"]
        >= config["amount_vs_avg_ratio_threshold"]
    )


def velocity_rule(transaction, config):
    return (
        transaction["txn_count_last_5min"]
        >= config["velocity_5min_threshold"]
    )


def impossible_travel_rule(transaction, config):
    return (
        transaction["distance_from_last_location_km"]
        >= config["impossible_travel_distance_km"]
        and
        transaction["time_since_last_txn_sec"]
        <= config["impossible_travel_time_sec"]
    )


def merchant_mismatch_rule(transaction, config):
    return (
        int(
            transaction["merchant_category_is_new_for_user"]
        )
        == 1
    )


print("========== RULE FUNCTIONS ==========")
print("Amount threshold rule:", callable(amount_threshold_rule))
print("Velocity rule:", callable(velocity_rule))
print("Impossible travel rule:", callable(impossible_travel_rule))
print("Merchant mismatch rule:", callable(merchant_mismatch_rule))
print("====================================")

========== RULE FUNCTIONS ==========
Amount threshold rule: True
Velocity rule: True
Impossible travel rule: True
Merchant mismatch rule: True


In [4]:
def evaluate_rules(transaction, config):
    required_fields = [
        "amount",
        "amount_vs_avg_ratio",
        "txn_count_last_5min",
        "time_since_last_txn_sec",
        "distance_from_last_location_km",
        "merchant_category_is_new_for_user"
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in transaction
    ]

    if missing_fields:
        raise ValueError(
            f"Missing rule fields: {missing_fields}"
        )

    flags = []

    if amount_threshold_rule(
        transaction,
        config
    ):
        flags.append("HIGH_AMOUNT")

    if velocity_rule(
        transaction,
        config
    ):
        flags.append("HIGH_VELOCITY")

    if impossible_travel_rule(
        transaction,
        config
    ):
        flags.append("IMPOSSIBLE_TRAVEL")

    if merchant_mismatch_rule(
        transaction,
        config
    ):
        flags.append("NEW_MERCHANT_CATEGORY")

    if not flags:
        rule_score = 0.0
    else:
        rule_score = sum(
            config["rule_weights"][flag]
            for flag in flags
        )

    rule_score = min(
        max(
            float(rule_score),
            0.0
        ),
        1.0
    )

    return {
        "rule_flags": flags,
        "rule_score": rule_score,
        "rules_triggered": len(flags)
    }


print("========== RULE ENGINE FUNCTION ==========")
print(
    "Complete rule evaluation function:",
    callable(evaluate_rules)
)
print("===========================================")

========== RULE ENGINE FUNCTION ==========
Complete rule evaluation function: True


In [5]:
normal_transaction = {
    "amount": 50.0,
    "amount_vs_avg_ratio": 1.1,
    "txn_count_last_5min": 1,
    "time_since_last_txn_sec": 3600.0,
    "distance_from_last_location_km": 5.0,
    "merchant_category_is_new_for_user": 0
}

normal_result = evaluate_rules(
    normal_transaction,
    RULE_CONFIG
)

print("========== NORMAL TRANSACTION TEST ==========")
print(normal_result)
print("No rules triggered:", len(normal_result["rule_flags"]) == 0)
print("Rule score:", normal_result["rule_score"])
print("=============================================")

========== NORMAL TRANSACTION TEST ==========
{'rule_flags': [], 'rule_score': 0.0, 'rules_triggered': 0}
No rules triggered: True
Rule score: 0.0


In [6]:
test_cases = {
    "HIGH_AMOUNT": {
        "amount": 500.0,
        "amount_vs_avg_ratio": 4.0,
        "txn_count_last_5min": 1,
        "time_since_last_txn_sec": 7200.0,
        "distance_from_last_location_km": 5.0,
        "merchant_category_is_new_for_user": 0
    },
    "HIGH_VELOCITY": {
        "amount": 50.0,
        "amount_vs_avg_ratio": 1.1,
        "txn_count_last_5min": 6,
        "time_since_last_txn_sec": 30.0,
        "distance_from_last_location_km": 5.0,
        "merchant_category_is_new_for_user": 0
    },
    "IMPOSSIBLE_TRAVEL": {
        "amount": 50.0,
        "amount_vs_avg_ratio": 1.1,
        "txn_count_last_5min": 1,
        "time_since_last_txn_sec": 300.0,
        "distance_from_last_location_km": 700.0,
        "merchant_category_is_new_for_user": 0
    },
    "NEW_MERCHANT_CATEGORY": {
        "amount": 50.0,
        "amount_vs_avg_ratio": 1.1,
        "txn_count_last_5min": 1,
        "time_since_last_txn_sec": 7200.0,
        "distance_from_last_location_km": 5.0,
        "merchant_category_is_new_for_user": 1
    }
}

individual_results = {}

for expected_rule, transaction in test_cases.items():
    result = evaluate_rules(
        transaction,
        RULE_CONFIG
    )
    individual_results[expected_rule] = result

print("========== INDIVIDUAL RULE TESTS ==========")

for expected_rule, result in individual_results.items():
    print()
    print("Expected:", expected_rule)
    print("Actual:", result)

print("===========================================")

========== INDIVIDUAL RULE TESTS ==========

Expected: HIGH_AMOUNT
Actual: {'rule_flags': ['HIGH_AMOUNT'], 'rule_score': 0.25, 'rules_triggered': 1}

Expected: HIGH_VELOCITY
Actual: {'rule_flags': ['HIGH_VELOCITY'], 'rule_score': 0.25, 'rules_triggered': 1}

Expected: IMPOSSIBLE_TRAVEL
Actual: {'rule_flags': ['IMPOSSIBLE_TRAVEL'], 'rule_score': 0.25, 'rules_triggered': 1}

Expected: NEW_MERCHANT_CATEGORY
Actual: {'rule_flags': ['NEW_MERCHANT_CATEGORY'], 'rule_score': 0.25, 'rules_triggered': 1}


In [7]:
expected_flag_mapping = {
    "HIGH_AMOUNT": "HIGH_AMOUNT",
    "HIGH_VELOCITY": "HIGH_VELOCITY",
    "IMPOSSIBLE_TRAVEL": "IMPOSSIBLE_TRAVEL",
    "NEW_MERCHANT_CATEGORY": "NEW_MERCHANT_CATEGORY"
}

individual_test_status = {}

for test_name, result in individual_results.items():
    expected_flag = expected_flag_mapping[test_name]

    individual_test_status[test_name] = (
        expected_flag in result["rule_flags"]
    )

print("========== INDIVIDUAL RULE VALIDATION ==========")

for test_name, status in individual_test_status.items():
    print(
        f"{test_name}:",
        status
    )

print()

print(
    "All individual rule tests passed:",
    all(
        individual_test_status.values()
    )
)

print("===============================================")

========== INDIVIDUAL RULE VALIDATION ==========
HIGH_AMOUNT: True
HIGH_VELOCITY: True
IMPOSSIBLE_TRAVEL: True
NEW_MERCHANT_CATEGORY: True

All individual rule tests passed: True


In [8]:
multi_rule_transaction = {
    "amount": 2000.0,
    "amount_vs_avg_ratio": 8.0,
    "txn_count_last_5min": 10,
    "time_since_last_txn_sec": 120.0,
    "distance_from_last_location_km": 1200.0,
    "merchant_category_is_new_for_user": 1
}

multi_rule_result = evaluate_rules(
    multi_rule_transaction,
    RULE_CONFIG
)

expected_multi_flags = {
    "HIGH_AMOUNT",
    "HIGH_VELOCITY",
    "IMPOSSIBLE_TRAVEL",
    "NEW_MERCHANT_CATEGORY"
}

actual_multi_flags = set(
    multi_rule_result["rule_flags"]
)

multi_rule_test_passed = (
    actual_multi_flags
    == expected_multi_flags
)

print("========== MULTI-RULE TEST ==========")
print("Result:", multi_rule_result)
print(
    "All four rules triggered:",
    multi_rule_test_passed
)
print("======================================")

========== MULTI-RULE TEST ==========
Result: {'rule_flags': ['HIGH_AMOUNT', 'HIGH_VELOCITY', 'IMPOSSIBLE_TRAVEL', 'NEW_MERCHANT_CATEGORY'], 'rule_score': 1.0, 'rules_triggered': 4}
All four rules triggered: True


In [9]:
rule_results = []

for _, row in test_df.iterrows():

    transaction = {
        column: row[column]
        for column in FEATURE_COLUMNS
    }

    result = evaluate_rules(
        transaction,
        RULE_CONFIG
    )

    rule_results.append(result)

rule_results_df = pd.DataFrame(
    rule_results
)

print("========== TEST DATA RULE EVALUATION ==========")
print(
    "Transactions evaluated:",
    len(rule_results_df)
)

print(
    "Rule scores valid:",
    rule_results_df["rule_score"]
    .between(0, 1)
    .all()
)

print(
    "Maximum rules triggered:",
    rule_results_df["rules_triggered"].max()
)

print("===============================================")

========== TEST DATA RULE EVALUATION ==========
Transactions evaluated: 555719
Rule scores valid: True
Maximum rules triggered: 2


In [10]:
rule_flag_counts = {
    "HIGH_AMOUNT": 0,
    "HIGH_VELOCITY": 0,
    "IMPOSSIBLE_TRAVEL": 0,
    "NEW_MERCHANT_CATEGORY": 0
}

for flags in rule_results_df["rule_flags"]:
    for flag in flags:
        rule_flag_counts[flag] += 1

rule_statistics = pd.DataFrame({
    "rule": list(rule_flag_counts.keys()),
    "trigger_count": list(rule_flag_counts.values())
})

rule_statistics["trigger_rate"] = (
    rule_statistics["trigger_count"]
    / len(rule_results_df)
)

print("========== RULE STATISTICS ==========")
print(
    rule_statistics.to_string(
        index=False
    )
)

print("=====================================")

========== RULE STATISTICS ==========
                 rule  trigger_count  trigger_rate
          HIGH_AMOUNT          20714      0.037274
        HIGH_VELOCITY              0      0.000000
    IMPOSSIBLE_TRAVEL              0      0.000000
NEW_MERCHANT_CATEGORY             83      0.000149


In [11]:
analysis_df = test_df[
    [
        "is_fraud"
    ]
].copy()

analysis_df["rule_score"] = (
    rule_results_df["rule_score"]
    .to_numpy()
)

analysis_df["rules_triggered"] = (
    rule_results_df["rules_triggered"]
    .to_numpy()
)

for rule_name in [
    "HIGH_AMOUNT",
    "HIGH_VELOCITY",
    "IMPOSSIBLE_TRAVEL",
    "NEW_MERCHANT_CATEGORY"
]:
    analysis_df[rule_name] = [
        int(
            rule_name in flags
        )
        for flags in rule_results_df["rule_flags"]
    ]

rule_fraud_summary = (
    analysis_df
    .groupby("is_fraud")[
        [
            "HIGH_AMOUNT",
            "HIGH_VELOCITY",
            "IMPOSSIBLE_TRAVEL",
            "NEW_MERCHANT_CATEGORY"
        ]
    ]
    .mean()
)

print("========== RULE / FRAUD ANALYSIS ==========")
print(
    rule_fraud_summary
)

print("============================================")

========== RULE / FRAUD ANALYSIS ==========
          HIGH_AMOUNT  HIGH_VELOCITY  IMPOSSIBLE_TRAVEL  NEW_MERCHANT_CATEGORY
is_fraud                                                                      
0            0.034758            0.0                0.0               0.000000
1            0.686713            0.0                0.0               0.038695


In [12]:
RULE_CONFIG_PATH = (
    MODELS_DIR
    / "rule_engine_config.json"
)

with open(
    RULE_CONFIG_PATH,
    "w"
) as file:
    json.dump(
        RULE_CONFIG,
        file,
        indent=4
    )

print("========== RULE CONFIG SAVED ==========")
print(
    "Path:",
    RULE_CONFIG_PATH
)

print(
    "File exists:",
    RULE_CONFIG_PATH.exists()
)

print("=======================================")

========== RULE CONFIG SAVED ==========
Path: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\rule_engine_config.json
File exists: True


In [13]:
RULE_ENGINE_CODE = '''from pathlib import Path
import json

SRC_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = SRC_DIR.parent
MODELS_DIR = PROJECT_ROOT / "models"

CONFIG_PATH = MODELS_DIR / "rule_engine_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Rule engine configuration not found: {CONFIG_PATH}"
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    RULE_CONFIG = json.load(file)


def amount_threshold_rule(transaction, config):
    return (
        float(transaction["amount_vs_avg_ratio"])
        >= float(
            config["amount_vs_avg_ratio_threshold"]
        )
    )


def velocity_rule(transaction, config):
    return (
        int(transaction["txn_count_last_5min"])
        >= int(
            config["velocity_5min_threshold"]
        )
    )


def impossible_travel_rule(transaction, config):
    return (
        float(
            transaction["distance_from_last_location_km"]
        )
        >= float(
            config["impossible_travel_distance_km"]
        )
        and
        float(
            transaction["time_since_last_txn_sec"]
        )
        <= float(
            config["impossible_travel_time_sec"]
        )
    )


def merchant_mismatch_rule(transaction, config):
    return (
        int(
            transaction[
                "merchant_category_is_new_for_user"
            ]
        )
        == 1
    )


def evaluate_rules(transaction):
    required_fields = [
        "amount",
        "amount_vs_avg_ratio",
        "txn_count_last_5min",
        "time_since_last_txn_sec",
        "distance_from_last_location_km",
        "merchant_category_is_new_for_user"
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in transaction
    ]

    if missing_fields:
        raise ValueError(
            f"Missing rule fields: {missing_fields}"
        )

    flags = []

    if amount_threshold_rule(
        transaction,
        RULE_CONFIG
    ):
        flags.append("HIGH_AMOUNT")

    if velocity_rule(
        transaction,
        RULE_CONFIG
    ):
        flags.append("HIGH_VELOCITY")

    if impossible_travel_rule(
        transaction,
        RULE_CONFIG
    ):
        flags.append("IMPOSSIBLE_TRAVEL")

    if merchant_mismatch_rule(
        transaction,
        RULE_CONFIG
    ):
        flags.append("NEW_MERCHANT_CATEGORY")

    if not flags:
        rule_score = 0.0
    else:
        rule_score = sum(
            float(
                RULE_CONFIG["rule_weights"][flag]
            )
            for flag in flags
        )

    rule_score = min(
        max(
            rule_score,
            0.0
        ),
        1.0
    )

    return {
        "rule_flags": flags,
        "rule_score": float(rule_score),
        "rules_triggered": len(flags)
    }
'''

RULE_ENGINE_PATH = (
    SRC_DIR
    / "rule_engine.py"
)

RULE_ENGINE_PATH.write_text(
    RULE_ENGINE_CODE,
    encoding="utf-8"
)

print("========== RULE ENGINE MODULE ==========")
print(
    "File:",
    RULE_ENGINE_PATH
)

print(
    "File exists:",
    RULE_ENGINE_PATH.exists()
)

print("=========================================")

========== RULE ENGINE MODULE ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\src\rule_engine.py
File exists: True


In [14]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rule_engine import evaluate_rules as production_evaluate_rules

production_normal_result = production_evaluate_rules(
    normal_transaction
)

production_multi_result = production_evaluate_rules(
    multi_rule_transaction
)

production_matches_notebook = (
    production_normal_result
    == normal_result
    and
    production_multi_result
    == multi_rule_result
)

print("========== PRODUCTION RULE ENGINE TEST ==========")
print(
    "Production normal result:",
    production_normal_result
)

print(
    "Production multi-rule result:",
    production_multi_result
)

print(
    "Production matches notebook:",
    production_matches_notebook
)

print("=================================================")

========== PRODUCTION RULE ENGINE TEST ==========
Production normal result: {'rule_flags': [], 'rule_score': 0.0, 'rules_triggered': 0}
Production multi-rule result: {'rule_flags': ['HIGH_AMOUNT', 'HIGH_VELOCITY', 'IMPOSSIBLE_TRAVEL', 'NEW_MERCHANT_CATEGORY'], 'rule_score': 1.0, 'rules_triggered': 4}
Production matches notebook: True


In [15]:
production_results = []

sample_size = min(
    100,
    len(test_df)
)

for _, row in test_df.head(sample_size).iterrows():

    transaction = {
        column: row[column]
        for column in FEATURE_COLUMNS
    }

    result = production_evaluate_rules(
        transaction
    )

    production_results.append(
        result
    )

production_results_df = pd.DataFrame(
    production_results
)

production_scores_valid = (
    production_results_df["rule_score"]
    .between(0, 1)
    .all()
)

production_flags_valid = all(
    isinstance(flags, list)
    for flags in production_results_df[
        "rule_flags"
    ]
)

print("========== PRODUCTION RULE BATCH TEST ==========")

print(
    "Transactions processed:",
    len(production_results_df)
)

print(
    "Rule scores valid:",
    production_scores_valid
)

print(
    "Rule flags valid:",
    production_flags_valid
)

print(
    "Batch test passed:",
    (
        len(production_results_df) == sample_size
        and
        production_scores_valid
        and
        production_flags_valid
    )
)

print("===============================================")

========== PRODUCTION RULE BATCH TEST ==========
Transactions processed: 100
Rule scores valid: True
Rule flags valid: True
Batch test passed: True


In [16]:
final_rule_engine_ready = all([
    len(individual_test_status) == 4,
    all(individual_test_status.values()),
    multi_rule_test_passed,
    RULE_CONFIG_PATH.exists(),
    RULE_ENGINE_PATH.exists(),
    production_matches_notebook,
    production_scores_valid,
    production_flags_valid,
    len(production_results_df) == sample_size
])

print()
print("================================================")
print("     STREAMSENTINEL V2 — PHASE 9 SUMMARY")
print("================================================")

print()

print("Rules implemented:")
print("1. HIGH_AMOUNT")
print("2. HIGH_VELOCITY")
print("3. IMPOSSIBLE_TRAVEL")
print("4. NEW_MERCHANT_CATEGORY")

print()

print(
    "Individual rule tests passed:",
    all(
        individual_test_status.values()
    )
)

print(
    "Multi-rule test passed:",
    multi_rule_test_passed
)

print(
    "Rule configuration saved:",
    RULE_CONFIG_PATH.exists()
)

print(
    "Production rule engine saved:",
    RULE_ENGINE_PATH.exists()
)

print(
    "Production module matches notebook:",
    production_matches_notebook
)

print(
    "Production batch test passed:",
    (
        production_scores_valid
        and
        production_flags_valid
        and
        len(production_results_df) == sample_size
    )
)

print()

print(
    "FINAL RULE ENGINE STATUS:",
    "READY"
    if final_rule_engine_ready
    else "NOT READY"
)

print(
    "Overall verification:",
    final_rule_engine_ready
)

print("================================================")


     STREAMSENTINEL V2 — PHASE 9 SUMMARY

Rules implemented:
1. HIGH_AMOUNT
2. HIGH_VELOCITY
3. IMPOSSIBLE_TRAVEL
4. NEW_MERCHANT_CATEGORY

Individual rule tests passed: True
Multi-rule test passed: True
Rule configuration saved: True
Production rule engine saved: True
Production module matches notebook: True
Production batch test passed: True

FINAL RULE ENGINE STATUS: READY
Overall verification: True
